# 01 — Dataset Exploration

Understand what data we have before training anything:
- Pothole dataset sizes and class distribution
- Traffic light dataset sizes
- Severity crop distribution (Low / Medium / High)
- Sample images with ground-truth bounding boxes
- Bounding box size distribution (informs severity thresholds)

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter

BASE_DIR = Path('..').resolve()
print('Project root:', BASE_DIR)

## 1. Detector Dataset Overview

In [ ]:
DETECTOR = BASE_DIR / 'data' / 'processed' / 'detector_yolo'

for split in ['train', 'val', 'test']:
    imgs = list((DETECTOR / 'images' / split).glob('*'))
    lbls = list((DETECTOR / 'labels' / split).glob('*.txt'))
    print(f'  {split:5s} → {len(imgs):5d} images  {len(lbls):5d} label files')

In [ ]:
# Count class distribution across training labels
# class 0 = pothole, class 1 = traffic_light
class_counts = Counter()
for lbl_path in (DETECTOR / 'labels' / 'train').glob('*.txt'):
    for line in lbl_path.read_text().strip().splitlines():
        if line:
            cls = int(line.split()[0])
            class_counts[cls] += 1

names = {0: 'pothole', 1: 'traffic_light'}
labels = [names[k] for k in sorted(class_counts)]
counts = [class_counts[k] for k in sorted(class_counts)]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts, color=['#e74c3c', '#2ecc71'])
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Bounding Box Count per Class (train split)', fontsize=13)
ax.set_ylabel('Number of boxes')
plt.tight_layout()
plt.show()
print('\nClass distribution:', dict(zip(labels, counts)))

## 2. Bounding Box Size Distribution

This helps validate the severity auto-labeling thresholds:
- `box_area / image_area < 0.03` → Low
- `0.03 – 0.12` → Medium
- `≥ 0.12` → High

In [ ]:
pothole_area_ratios = []

for lbl_path in (DETECTOR / 'labels' / 'train').glob('*.txt'):
    if 'pothole' not in lbl_path.stem and not lbl_path.stem.startswith('chitholian') \
       and not lbl_path.stem.startswith('kaggle'):
        continue
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if not parts or int(parts[0]) != 0:
            continue
        # YOLO format: cls xc yc bw bh (normalized)
        bw, bh = float(parts[3]), float(parts[4])
        pothole_area_ratios.append(bw * bh)

print(f'Pothole boxes analysed: {len(pothole_area_ratios)}')

low    = sum(1 for r in pothole_area_ratios if r < 0.03)
medium = sum(1 for r in pothole_area_ratios if 0.03 <= r < 0.12)
high   = sum(1 for r in pothole_area_ratios if r >= 0.12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(pothole_area_ratios, bins=50, color='#e74c3c', edgecolor='white')
axes[0].axvline(0.03, color='orange', linestyle='--', label='Low/Med threshold (0.03)')
axes[0].axvline(0.12, color='red',    linestyle='--', label='Med/High threshold (0.12)')
axes[0].set_title('Pothole Box Area Distribution')
axes[0].set_xlabel('box_area / image_area (normalised)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Pie
axes[1].pie([low, medium, high], labels=['Low', 'Medium', 'High'],
            autopct='%1.1f%%', colors=['#27ae60', '#f39c12', '#e74c3c'],
            startangle=90)
axes[1].set_title('Auto-labelled Severity Distribution')

plt.tight_layout()
plt.show()
print(f'Low: {low}  Medium: {medium}  High: {high}')

## 3. Severity Crops Distribution

In [ ]:
SEVERITY = BASE_DIR / 'data' / 'processed' / 'severity_crops'
classes  = ['Low', 'Medium', 'High']
colors   = ['#27ae60', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(classes))
width = 0.25

for i, split in enumerate(['train', 'val', 'test']):
    counts = []
    for cls in classes:
        d = SEVERITY / split / cls
        counts.append(len(list(d.glob('*'))) if d.exists() else 0)
    ax.bar(x + i * width, counts, width, label=split)

ax.set_xticks(x + width)
ax.set_xticklabels(classes)
ax.set_title('Severity Crop Counts per Split')
ax.set_ylabel('Number of crops')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Sample Images — Pothole

In [ ]:
def show_yolo_samples(images_dir, labels_dir, class_names, n=6, title=''):
    img_paths = sorted(images_dir.glob('*'))[:n]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    colors_map = {0: (255, 80, 80), 1: (80, 200, 80)}

    for ax, img_path in zip(axes, img_paths):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl_path = labels_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
                c = colors_map.get(cls, (200, 200, 200))
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                         linewidth=2, edgecolor=[v/255 for v in c],
                                         facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, max(y1-4, 0), class_names[cls],
                        color=[v/255 for v in c], fontsize=8, fontweight='bold')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(img_path.name[:30], fontsize=7)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_yolo_samples(
    DETECTOR / 'images' / 'train',
    DETECTOR / 'labels' / 'train',
    class_names=['pothole', 'traffic_light'],
    title='Sample Training Images — Detector Dataset'
)

## 5. Sample Severity Crops

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
severity_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}

for row, sev in enumerate(['Low', 'Medium', 'High']):
    crop_paths = sorted((SEVERITY / 'train' / sev).glob('*'))[:5]
    for col, crop_path in enumerate(crop_paths):
        img = cv2.cvtColor(cv2.imread(str(crop_path)), cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        axes[row][col].set_title(sev, color=severity_colors[sev],
                                 fontsize=9, fontweight='bold')

plt.suptitle('Severity Crop Samples (Low / Medium / High)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print('=== Dataset Summary ===')
for split in ['train', 'val', 'test']:
    n = len(list((DETECTOR / 'images' / split).glob('*')))
    print(f'  Detector {split:5s}: {n} images')
print()
for split in ['train', 'val', 'test']:
    for cls in ['Low', 'Medium', 'High']:
        d = SEVERITY / split / cls
        n = len(list(d.glob('*'))) if d.exists() else 0
        print(f'  Severity {split:5s}/{cls:6s}: {n} crops')
print()
print('Ready to train: python src/train.py --stage all --aug')